# **SAE Lens: Basics of SAE in  Python**


SAE Lens — новый фреймворк и его функционал полностью посвящен SAE. Из-за того, что область только начинает развиваться, он может казаться немного пустым, но тем не менее библиотека многообещающая с точки зрения будущего.

В этой практике вы достигнете двух целей:


1.   Понять, как выглядит SAE в Python;
2.   Воспроизвести workflow анализа признаков, изученных SAE;

Успешного прохождения!



## **Set Up**
Подготовка окружения. При выборе среды выбирайте CPU, так как для этой практики можно обойтись без дополнительных вычислительных ресурсов.


In [ ]:
!pip install transformer-lens==3.9.0 plotly -q

In [ ]:
!pip install sae-lens==6.54.0 -q

****
**После выполнения ячеек выше, перезапустите ноутбук, прежде чем приступить к заданиям.**
****

In [ ]:
COLAB = True

# Стандартные библиотеки
import os
import torch
from tqdm import tqdm
import plotly.express as px
import pandas as pd

# Для экономии времени
#torch.set_grad_enabled(False)


if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {device}")

**Убедитесь, что версия библиотеки `sae_lens` >= 6.0.0.**

In [ ]:
!pip show sae-lens

# **Работа с автокодировщиками**

Все автокодировщики — модели, так что работа с ними – такая же, как и с другими моделями.

Например, когда у вас есть задача изучить какую-то модель, вы можете сначала поискать SAE *уже обученный на интересующей вас модели*. Это можно сделать либо через поиск по моделям в [HF](https://huggingface.co/models?search=SAE) 🤗, либо используя список обученных SAE в SAE Lens.


**Вопрос. Сколько разных SAE можно обучить на одной модели?**

`Выберите ответ на stepik`

In [ ]:
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory

df = pd.DataFrame.from_records(
    {k: v.__dict__ for k, v in get_pretrained_saes_directory().items()}
).T.reset_index(drop=True)

df.head()

Устройство полученной таблицы следующее:

- `release` — релиз, отражающий конкретный подход к SAE. Например:
  - gemma-scope-2b-pt-att
    - gemma-scope- — SAE, относящиеся к проекту [Gemma-Scope](https://huggingface.co/google/gemma-scope)
    - 2b-pt- — SAE обучены на base model Gemma v2 2B
    - att:  SAE обучены на выходах слоя attention

- `repo_id` — репозиторий 🤗, где хранятся модели. Часто там еще и есть исчерпывающая карточка — всегда полезно смотреть.
- `model` — модель, к которой относится автокодировщик
- `saes_map` — список всех SAE данного релиза
- `neuronpedia_id` — заполнено, если SAE есть на Neuronpedia
- `conversion_func`, `expected_var_explained`, `expected_l0`, `config_overrides` — рабочие столбцы.

Сейчас нет какого-то золотого стандарта по названию SAE в библиотеке, поэтому таблица может быть хаотичной. Но пример хороших и информативных названий ниже, надеюсь, поможет вам при загрузке своих моделей.

In [ ]:
print("SAEs in the GPT2 Small Resid Pre release")
for k, v in df.loc[df.release == "gpt2-small-res-jb", "saes_map"].values[0].items():
    print(f"SAE id: {k} for hook point: {v}")

print("-" * 50)
print("SAEs in the feature splitting release")
for k, v in (
    df.loc[df.release == "gpt2-small-res-jb-feature-splitting", "saes_map"]
    .values[0]
    .items()
):
    print(f"SAE id: {k} for hook point: {v}")

print("-" * 50)
print("SAEs in the Gemma base model release")
for k, v in df.loc[df.release == "gemma-2b-res-jb", "saes_map"].values[0].items():
    print(f"SAE id: {k} for hook point: {v}")

# **Работа с парой SAE-модель**

Загрузим пару SAE-Модель и еще немного разберемся со стандартами названий. А именно, с постфиксами `pre`, `post`, `attn`, `resid` в коде.


In [ ]:
from sae_lens import SAE, HookedSAETransformer
from transformers import GPT2Tokenizer, GPT2Model
import torch

# Для сравнения с hooked моделью, загрузим еще маленький GPT дообученный на португальском
hooked_model = HookedSAETransformer.from_pretrained("gpt2-small", device=device)
basic_model = GPT2Model.from_pretrained("pierreguillou/gpt2-small-portuguese").to(device)

sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",  # release
    sae_id="blocks.7.hook_resid_pre",  # SAE id
    device=device,

)

sae_model = sae  # в sae-lens 6 SAE.from_pretrained возвращает только SAE, без кортежа
sae_cfg = sae.cfg.to_dict()  # конфиг SAE в виде словаря

Выше использован SAE из блока 7 (`blocks.7`), обученный с помощью хука на residual stream для токенов $x_{pre}$ (`hook_resid_pre`). Найдем эту точку в модели.  Прежде, вспомним структуру трансформации эмбеддинга при проходе через конкретный блок.

`x_pre ──► Attention ──► x_mid ──► MLP ──► x_post (= x_pre of next layer)`

Теперь посмотрим на блок 7 в коде для базового и hooked GPT2.

In [ ]:
hooked_model.blocks[7]

In [ ]:
basic_model.h[7]

По названиям хуков можно локализовать, выход какой точки блока они сохраняют.

**Задание. Сопоставьте хук и точку в модели на stepik.**

1. hook_normalized
2. hook_q_input
3. hook_resid_mid
4. hook_resid_post

i. Входной эмбеддинг перед домножением на матрицу $Q$ \\
ii. Представление после attention	до MLP \\
iii. Выход блока после всех преобразований \\
iv. Выход после layernorm \\

### **SAE**
Теперь посмотрим, как выглядит SAE.

In [ ]:
sae_model # модель, также обвешанная хуками

In [ ]:
sae_cfg # полный конфиг модели

In [ ]:
sae_model.cfg.__dict__ # полный конфиг модели в более сжатом виде

<details>
<summary><b>Сноска с описанием параметров</b></summary>

1. `architecture`: Определяет тип архитектуры SAE. В данном случае стандартная архитектура кодировщик и декодировщик с промежуточными активациями.
2. `d_in`: Размерность входа в SAE, здесь 768.
3. `d_sae`: Размер скрытого слоя SAE, здесь 24576. Соответствует числу возможных активаций признаков.
5. `apply_b_dec_to_input`: Определяет, применяется ли смещение декодера ко входу.
7. `context_size`: Размер контекстного окна, здесь он равен 128 токенам. Оказалось, что SAE, обученные на небольших активациях из коротких подсказок, [часто работают хуже на длинных](https://www.lesswrong.com/posts/baJyjpktzmcmRfosq/stitching-saes-of-different-sizes).
8. `model_name`: Название используемой модели, здесь `'gpt2-small'`. [Это допустимое имя модели в TransformerLens](https://transformerlensorg.github.io/TransformerLens/generated/model_properties_table.html).
9. `hook_name`: Указывает конкретный хук в модели, к которому подключается SAE.
10. `hook_head_index`: Определяет, к какой голове внимания подключаться. Здесь не используется, поскольку речь идет о SAE, работающем с остаточным потоком (residual stream).
11. `prepend_bos`: Добавлять ли токен начала последовательности в начало. Здесь установлено в `True`.
12. `dataset_path`: Путь к датасету, используемому для обучения или оценки. Может быть локальным или из HuggingFace.
14. `normalize_activations`: Метод нормализации активаций. В этой конфигурации — `'none'`, то есть нормализация не применяется.
15. `dtype`: Тип данных для операций с тензорами. Здесь — 32-битное число с плавающей точкой.
16. `device`: Указывает вычислительное устройство (например, CPU или GPU).
17. `sae_lens_training_version`: Версия SAE Lens, использованная для обучения.
18. `activation_fn_kwargs`: Дополнительные параметры для функции активации. Например, если выбрана функция `topk`, здесь можно указать значение `k`.
19. `neuronpedia_id` — id на Neuronpedia. Если заполнено — то у SAE есть интерактивная визуализация на упомянутом [ресурсе](https://www.neuronpedia.org/).  
</details>

### **Признаки в SAE,  визуализация с Neuronpedia**


#### **Случайные признаки**


На примере удобной интеграции с Neuronpedia давайте посмотрим на признаки в SAE. Напомню, что в идеале для каждого входного примера SAE должен давать разреженное представление, то есть активировать $n$ признаков, так что $n$ сильно меньше чем $d_{sae}$.

Каждый признак представляет собой нейрон. Разметка нейронов, как признаков, происходит эмпирически.

Гипотетический пример: если при повторных прогонах тестовых данных нейрон $f$ с номером $x$ ($f_x$) всегда активируется на словах, связанных с котиками, то мы присваиваем ему признак — фразы, относящиеся к кошкам.

Теперь посмотрим на пример реальный.

In [ ]:
from IPython.display import IFrame

# Посмотрим на признак со случайным индексом
feature_idx = 3777

html_template = "https://neuronpedia.org/{}/{}/{}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"


def get_dashboard_html(sae_release="gpt2-small", sae_id="7-res-jb", feature_idx=0):
    return html_template.format(sae_release, sae_id, feature_idx)


html = get_dashboard_html(
    sae_release="gpt2-small", sae_id="7-res-jb", feature_idx=feature_idx
)
IFrame(html, width=1200, height=600)

Разметка признаков для каждого SAE проводилась либо человеком, либо объясняющей моделью: у признака 3777 на Neuronpedia два описания, оба от объясняющих моделей (gpt-3.5-turbo и gpt-4o-mini). Компоненты полученного дашборда позволяют проанализировать:

1. Описание признака (верхний левый угол, Feature Description) — *phrases related to anticipation or holding back*.

2. Графики логитов (левый угол, Logit Plots) — токены словаря, логиты которых направление признака в декодере сильнее всего повышает (positive) и понижает (negative).

3. Графики плотностей (правый угол, Activations Density Plot):
 - верхний график — гистограмма ненулевых активаций признака на случайной выборке текстов: по оси x — сила активации признака, по оси y — число токенов;
 - нижний график — гистограмма вкладов признака в логиты всех токенов словаря GPT-2: отрицательные значения у токенов, которые признак подавляет, положительные — у токенов, которые он продвигает.

4. Поле для тестирования активации признака — Test Activation. Здесь можно ввести свой текст и оценить реакцию признака на него.

5. Примеры с наиболее сильными активациями признака — Top Activations, после текста. На них можно анализировать предложения и место, на котором загорается признак.

**Вопрос. На основе графиков выберите верные утверждения на stepik:**

- Нижний график плотностей (вклады признака в логиты токенов) одногорбый: пик около нуля, спад в обе стороны. По форме он близок к нормальному распределению;
- Самый большой по модулю отрицательный вклад в логиты (Negative Logits) больше самого большого положительного (Positive Logits);
- Чаще всего ненулевые активации признака принимают значения не больше 5: самые высокие столбцы верхнего графика стоят в начале оси;
- Первые три столбца верхнего графика лежат левее активации 3;
- Все три первых столбца верхнего графика лежат левее активации 1

#### **Поиск признаков на заданную тему**


Для некоторых моделей в SAE Lens есть разметка признаков, полученная аннотированием, при помощи другой модели. Используя эту разметку, можно попробовать поискать признаки на заданную тему.

Проделаем это на примере признаков из модели GEMMA.

In [ ]:
import requests
import gzip
import json
import re

# Эндпоинт Neuronpedia /api/explanation/export удален. Выгрузки описаний признаков лежат
# в открытом S3-хранилище neuronpedia-datasets: v1/<модель>/<SAE>/explanations/*.jsonl.gz
def load_explanations(model_id, sae_id):
    base = "https://neuronpedia-datasets.s3.us-east-1.amazonaws.com"
    listing = requests.get(base, params={"list-type": "2", "prefix": f"v1/{model_id}/{sae_id}/explanations/"}).text
    rows = []
    for key in re.findall(r"<Key>([^<]+\.jsonl\.gz)</Key>", listing):
        raw = gzip.decompress(requests.get(f"{base}/{key}").content).decode()
        rows += [{k: v for k, v in json.loads(line).items() if k != "embedding"} for line in raw.splitlines()]
    return rows


#data = load_explanations("gpt2-small", "7-res-jb")
data = load_explanations("gemma-2-2b", "20-gemmascope-res-16k")

In [ ]:
# convert to pandas
explanations_df = pd.DataFrame(data)

# rename index to "feature"
explanations_df.rename(columns={"index": "feature"}, inplace=True)

explanations_df["description"] = explanations_df["description"].apply(
    lambda x: x.lower()
)
explanations_df.head()

**Вопрос. В SAE `20-gemmascope-res-16k` модели gemma-2-2b 16384 признака, номера от 0 до 16383. У скольких из них в полученной выгрузке нет ни одного описания?**

Подсказка: в выгрузке одна строка на одно описание, номер признака стоит в столбце `feature`.

In [ ]:
 # Ваш код здесь

Посмотрим на нейроны, кодирующие признаки, связанные с фруктами.

Таблица для gemma может оказаться пустой: сейчас признаки этого SAE на Neuronpedia размечены заново одной моделью, и описаний со словом fruits среди них нет. Для GPT-2 ниже такие описания есть.

In [ ]:
fruits_features = explanations_df.loc[explanations_df.description.str.contains(" fruits")]
fruits_features

**Проделайте то же самое для SAE `7-res-jb` модели GPT2. Получите таблицу и найдите признаки, в описании которых есть слово fruits. Сколько признаков SAE получилось?**

In [ ]:
 # Ваш код здесь

In [ ]:
fruits_features = explanations_df.loc[explanations_df.description.str.contains(" fruits")]
fruits_features

In [ ]:
# После нахождения признаков, запустите ячейку, чтобы увидеть дашборд для одного из них
html = get_dashboard_html(
    sae_release="gpt2-small",
    sae_id="7-res-jb",
    feature_idx=fruits_features.feature.values[0],
)
IFrame(html, width=1200, height=600)

### **Получение признаков на основе SAE**

В качестве последнего шага, посмотрим, как извлекать признаки, используя обученный SAE с размеченными (или нет) признаками.


In [ ]:
from transformer_lens.utils import test_prompt

prompt = "The best animal of the world is a"
answer = "cat"

# Посмотрим, как модель прогнозирует завершение предложения
test_prompt(prompt, answer, hooked_model)

### **Прогон модели с SAE**

В завершение посмотрим, как может выглядеть прогон модели с SAE. Вспомним, как выглядит класс SAE в SAE Lens.


In [ ]:
sae

Обратите внимание — SAE тоже обвешан хуками. Они позволяют анализировать проходы также в разных точках SAE. И это — очень полезно при интерпретации. А именно:

- `hook_sae_input` — вход ($input$)в SAE. То есть эмбеддинг из модели, подаваемый на вход автокодировщику. Размерность: `[batch, seq_len, d_model]`;

- `hook_sae_acts_pre` — sparse-представление до ReLU— результат умножения входа на весовую матрицу энкодера.
Формально, при `apply_b_dec_to_input=True`, как у этого SAE: $$acts_{pre} = (x - b_{dec})W_{enc} + b_{enc}$$ Размерность: `[batch, seq_len, d_sae]`;

- `hook_sae_acts_post` — sparse-представление после ReLU. Именно это обычно берется за интерпретируемое представление. Формально: $$acts_{post} = ReLU(acts_{pre})$$ Размерность: `[batch, seq_len, d_sae]`;


- `hook_sae_recons` —  реконструкция после SAE, но до обратной нормализации и reshape, если они применяются. «Сырые» восстановленные активации. Формально:
$$acts_{post}W_{dec} + b_{dec}$$ Размерность: `[batch, seq_len, d_model]`;


- `hook_sae_error` — разница между реконструкцией и оригинальной активацией, то есть формально: $$error = input - reconstruction$$ Размерность: `[batch, seq_len, d_model]`;

- `hook_sae_output` —  окончательные восстановленные активации после всех шагов (обратная нормализация, reshaping). При use_error_term=True сюда добавляется ошибка реконструкции (hook_sae_error).
Формально: $$recons = acts_{post}W_{dec} + b_{dec} + error$$ Размерность: `[batch, seq_len, d_model]`;



**Параметр `use_error_term`**.

При прогоне модели с SAE после SAE на вход следующего слоя передаются реконструкции автокодировщика. В идеальном случае (если автокодировщик отлично обучен) это не должно влиять на поведение модели, однако в реальности SAE не восстанавливают активации в точности.  Поэтому, чтобы сохранять производительность модели, к реконструкции добавляют ошибку $error$, чтобы получить тот же эмбеддинг и ту же производительность.

В SAE Lens эта логика реализована за счет атрибута `sae.use_error_term`. Если установить его равным `True`, то реконструкция после SAE будет учитывать ошибку приближения.

In [ ]:
sae.use_error_term  # Если использовать error term, то дальше по модели будет пущен оригинальный эмбеддинг. Оставьте False

In [ ]:
# Получим представления SAE, прогнав модель с хуками и SAE
_, cache = hooked_model.run_with_cache_with_saes(prompt, saes=[sae])

k, v = list(cache.items())[0]
print(k, v.shape)

Каждое собранное значение в слое представляет собой представление из модели по всем хукам. Проанализируйте кэш и проверьте, сколько представлений связано с SAE.

In [ ]:
for k, v in cache.items():
  if 'sae' in k:
    print(k, v.shape)

Представлений получилось меньше, чем хуков в SAE. Это связано с параметром `use_error_term` — не нужное не вычислялось и, соответственно, не регистрировалось.

Теперь посмотрим, какие признаки активировались на седьмом слое и как выглядело представление.

In [ ]:
px.line(
    cache["blocks.7.hook_resid_pre.hook_sae_acts_post"][0, -1, :].cpu().numpy(),
    title="Feature activations at the final token position",
    labels={"index": "Feature", "value": "Activation"},
).show()

# Топ 5 токенов
vals, inds = torch.topk(
    cache["blocks.7.hook_resid_pre.hook_sae_acts_post"][0, -1, :], 5
)
for val, ind in zip(vals, inds):
    print(f"Feature {ind} fired {val:.2f}")
    html = get_dashboard_html(
        sae_release="gpt2-small", sae_id="7-res-jb", feature_idx=ind
    )
    display(IFrame(html, width=1200, height=300))

Ни одного признака, связанного с кошками! И это не аномальное поведение. Вспомните урок по logit lens — модели формируют ответ постепенно, обрабатывая input layer by layer.

## **Заключение**

На этой практике вы базово поработали с обученными SAE и проанализировали спектр задач, который можно решить с обученным автокодировщиком. Важно здесь заострить внимание на следующих деталях:

- SAE Lens — предоставляет удобную обертку для работы с SAE. Но это не отменяет того, что SAE — самостоятельные модели и вы можете писать свою конфигурацию под свои нужды, в том числе опираясь на удобную кодовую базу библиотеки.
- Анализ признаков для уже обученного SAE требует разметки. Ее можно автоматизировать — способы мы обсудим далее по практике и теории.
- В процессе анализа изученных признаков можно применять разные сценарии. А именно — анализировать немного измененные промт для оценки того, как меняется активация, смотреть на примеры и токена, где активация фичи максимальна. Кроме того, на основе SAE можно делать стиринг.

Обученный SAE — начало большого поля для творчества. Изучайте и исследуйте!